# 00 — Figure 1 | PPDT Pioneer Panel Library

Phylogenetic diversity of the 11 PPDT pioneer genomes (Figures 1a–b) and insert-level
library characterisation from the merged short-read × long-read library (Figures 1c–f).

## Configuration

In [ ]:
from pathlib import Path

# --- data directory (populate yourself -- see README's Data section) ---
DATA_DIR = Path("../data")
RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# --- phylogeny resources (GTDB release r232) ---
TAX_PATH   = DATA_DIR / "bac120_taxonomy_r232.tsv"
SP_PATH    = DATA_DIR / "sp_clusters_r232.tsv"
TREE_PATH  = DATA_DIR / "bac120_r232.tree"

# --- library characterisation ---
MERGED_PARQUET_PATH = DATA_DIR / "library_characterization_insert_data.parquet"
GENE_INFO_PATH = DATA_DIR / "gene_fitness_results_with_annotations.parquet"

# --- 11 PPDT pioneer genomes ---
PPDT_GENOMES = {
    "Bacillus_subtilis_PY79_GCF_023521615.1":                   "Gram positive model organism",
    "Colwellia_psychrerythraea_34H_BAA-681_GCF_000012325.1":    "Cold-loving (psychrophile)",
    "Cupriavidus_necator_H16_GCF_004798725.1":                  "CO2-fixing bioplastic producer",
    "Deinococcus_radiodurans_ATCC_13939_GCF_020546685.1":       "Extremely radiation-resistant",
    "Escherichia_coli_K-12_substr._MG1655_GCF_000005845.2":     "Model gram-negative, library host",
    "Geodermatophilus_obscurus_DSM_43160_GCF_000025345.1":      "Desiccation- and oxidation-resistant",
    "Halomonas_elongata_DSM_2581_GCF_000196875.2":              "Salt-loving (halophile)",
    "Klebsiella_variicola_DSM_15968_GCF_000828055.2":           "Nitrogen-fixing soil bacteria",
    "Planococcus_halocryophilus_DSM_24743_GCF_001687585.2":     "Cold- and salt-loving",
    "Pseudomonas_putida_KT2440_GCF_045571375.1":                "Metabolically versatile chassis",
    "Rubrobacter_radiotolerans_ATCC_51242_GCF_033842665.1":     "Heat- and radiation-tolerant",
}


## Imports

In [ ]:
import re
import copy
import time

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import seaborn as sns
import dendropy
from great_tables import GT

---
# Figure 1a — Phylogenetic tree of PPDT pioneer genomes

Map the 11 PPAT pioneer genomes to their GTDB r232 species assignments and representative
genomes, then extract their placement in the GTDB bac120 tree to build a pruned phylogeny
and all-pairs patristic distance matrix.

## Build pioneer_genome → GTDB mapping

In [ ]:
# --- step 1: GTDB taxonomy lookup ---
tax = pd.read_csv(
    TAX_PATH, sep="\t", header=None,
    names=["accession", "gtdb_taxonomy"],
)
tax_by_acc = tax.set_index("accession")["gtdb_taxonomy"]

# --- step 2: species → representative lookup ---
sp = pd.read_csv(
    SP_PATH, sep="\t", usecols=[0, 1],
    names=["representative_genome", "gtdb_species"], header=0,
)
rep_by_species = sp.set_index("gtdb_species")["representative_genome"]

# --- build mapping table ---
TAXONOMY_FIELDS = {
    "gtdb_phylum":  1,
    "gtdb_class":   2,
    "gtdb_order":   3,
    "gtdb_family":  4,
    "gtdb_species": 6,
}

rows = []
for genome in PPDT_GENOMES:
    gcf = re.search(r"(GCF_[\d.]+)", genome).group(1)
    taxonomy = tax_by_acc.get(f"RS_{gcf}")
    parts = taxonomy.split(";") if taxonomy else []
    row = dict(pioneer_genome=genome, gcf_accession=gcf)
    for field, idx in TAXONOMY_FIELDS.items():
        row[field] = parts[idx].strip() if len(parts) > idx else None
    row["representative_genome"] = rep_by_species.get(row["gtdb_species"]) if row["gtdb_species"] else None
    rows.append(row)

genome_map = pd.DataFrame(rows)
genome_map["description"] = genome_map["pioneer_genome"].map(PPDT_GENOMES)

missing = genome_map[genome_map["representative_genome"].isna()]
assert missing.empty, f"Missing GTDB mapping for: {missing['pioneer_genome'].tolist()}"
print(f"All {len(genome_map)} genomes mapped successfully.")

In [ ]:
genome_map["is_gtdb_rep"] = genome_map.apply(
    lambda r: r["representative_genome"] == f"RS_{r['gcf_accession']}", axis=1
)

display(genome_map[[
    "pioneer_genome", "gcf_accession", "description",
    "gtdb_phylum", "gtdb_class", "gtdb_order", "gtdb_family",
    "gtdb_species", "representative_genome", "is_gtdb_rep",
]])

## Load GTDB r232 bac120 tree

In [ ]:
t0 = time.time()
tree = dendropy.Tree.get(
    path=str(TREE_PATH),
    schema="newick",
    preserve_underscores=True,
)
tree.is_rooted = True
print(f"Tree loaded in {time.time() - t0:.1f}s  ({len(tree.taxon_namespace):,} taxa)")

## Prune tree to 11 pioneer taxa

In [ ]:
tree_labels = {t.label for t in tree.taxon_namespace}
for _, row in genome_map.iterrows():
    rep = row["representative_genome"]
    status = "OK" if rep in tree_labels else "MISSING"
    print(f"[{status}] {row['pioneer_genome'].split('_GCF_')[0]}: {rep}")

missing_in_tree = genome_map[~genome_map["representative_genome"].isin(tree_labels)]
assert missing_in_tree.empty, f"Missing from tree: {missing_in_tree['representative_genome'].tolist()}"

rep_labels = genome_map["representative_genome"].tolist()
pruned = copy.deepcopy(tree)
pruned.retain_taxa_with_labels(rep_labels)
print(f"\nLeaves after pruning: {len(list(pruned.leaf_node_iter()))}") 

## Figure 1a — Plot pruned phylogenetic tree

In [ ]:
PHYLUM_COLORS = {
    "p__Pseudomonadota": "#4472C4",
    "p__Bacillota":      "#70AD47",
    "p__Actinomycetota": "#ED7D31",
    "p__Deinococcota":   "#C00000",
}

rep_to_label       = {row["representative_genome"]: row["pioneer_genome"].split("_GCF_")[0].replace("_", " ") for _, row in genome_map.iterrows()}
rep_to_phylum      = {row["representative_genome"]: row["gtdb_phylum"] for _, row in genome_map.iterrows()}
rep_to_description = {row["representative_genome"]: row["description"] for _, row in genome_map.iterrows()}


def get_node_coords(tree):
    leaves   = list(tree.leaf_node_iter())
    leaf_y   = {leaf: i for i, leaf in enumerate(leaves)}
    x_pos, y_pos = {}, {}

    def x_from_root(node):
        if node not in x_pos:
            edge_len = node.edge.length or 0.0
            parent   = node.parent_node
            x_pos[node] = (x_from_root(parent) if parent else 0.0) + edge_len
        return x_pos[node]

    for node in tree.preorder_node_iter():
        x_from_root(node)

    def y_coord(node):
        if node not in y_pos:
            if node.is_leaf():
                y_pos[node] = leaf_y[node]
            else:
                children_y = [y_coord(c) for c in node.child_nodes()]
                y_pos[node] = np.mean(children_y)
        return y_pos[node]

    for node in tree.postorder_node_iter():
        y_coord(node)

    return x_pos, y_pos, leaves


x_pos, y_pos, leaves = get_node_coords(pruned)

fig, ax = plt.subplots(figsize=(6, 4))

for node in pruned.preorder_node_iter():
    parent = node.parent_node
    if parent is not None:
        ax.plot([x_pos[parent], x_pos[node]], [y_pos[node], y_pos[node]], color="#333333", lw=1.2)

for node in pruned.preorder_node_iter():
    children = list(node.child_nodes())
    if children:
        ys = [y_pos[c] for c in children]
        ax.plot([x_pos[node], x_pos[node]], [min(ys), max(ys)], color="#333333", lw=1.2)

x_max = max(x_pos.values())
for leaf in leaves:
    rep   = leaf.taxon.label
    label = rep_to_label.get(rep, rep)
    desc  = rep_to_description.get(rep, "")
    color = PHYLUM_COLORS.get(rep_to_phylum.get(rep, ""), "#666666")
    y     = y_pos[leaf]
    ax.text(x_max * 1.01, y + 0.05, label, va="bottom", ha="left", fontsize=8, color=color, style="italic")
    ax.text(x_max * 1.01, y - 0.05, f"({desc})", va="top", ha="left", fontsize=7, color="#666666", style="italic")

ax.legend(
    handles=[mpatches.Patch(color=c, label=p.replace("p__", "")) for p, c in PHYLUM_COLORS.items()],
    title="GTDB phylum", title_fontsize=8, fontsize=7, loc="lower left",
)
ax.set_xlim(left=0, right=x_max * 1.1)
ax.set_ylim(-0.7, len(leaves) - 0.3)
ax.set_xlabel("GTDB r232 distance to root", fontsize=10)
ax.set_yticks([])
ax.spines[["top", "right", "left"]].set_visible(False)
plt.tight_layout()
fig.savefig(RESULTS_DIR / "figure1a_phylogeny_pioneer_genomes.pdf", bbox_inches="tight")
plt.show()

## Patristic distance to E. coli

In [ ]:
def patristic_distance(t, label_a, label_b):
    taxon_a = t.taxon_namespace.get_taxon(label=label_a)
    taxon_b = t.taxon_namespace.get_taxon(label=label_b)
    mrca    = t.mrca(taxa=[taxon_a, taxon_b])
    def dist_to_mrca(node, stop):
        d = 0.0
        while node is not stop:
            d += node.edge.length or 0.0
            node = node.parent_node
        return d
    return (dist_to_mrca(t.find_node_with_taxon_label(label_a), mrca) +
            dist_to_mrca(t.find_node_with_taxon_label(label_b), mrca))


ecoli_rep = genome_map.loc[
    genome_map["pioneer_genome"].str.startswith("Escherichia_coli"),
    "representative_genome",
].iloc[0]

dist_rows = []
for _, row in genome_map.iterrows():
    d = patristic_distance(pruned, row["representative_genome"], ecoli_rep)
    dist_rows.append({"pioneer_genome": row["pioneer_genome"], "distance_to_ecoli": d})

combined = genome_map.merge(pd.DataFrame(dist_rows), on="pioneer_genome").sort_values("distance_to_ecoli")
combined.to_csv(RESULTS_DIR / "pioneer_genome_gtdb_phylogeny.tsv", sep="\t", index=False)
display(combined[["pioneer_genome", "gtdb_phylum", "gtdb_species", "representative_genome", "is_gtdb_rep", "distance_to_ecoli"]])

---
# Library Characterisation

All insert-level figures below use the merged short-read × long-read PPDT library produced
by `library-selection-integration`. `gene_ids_fully_covered` lists the gene IDs whose entire
coding sequence is spanned by each insert.

## Load merged library data and gene information

In [ ]:
merged = pd.read_parquet(MERGED_PARQUET_PATH)
merged["insert_type"] = merged["insert_type"].fillna("unknown")
print(f"Merged shape: {merged.shape}")
print(merged["insert_type"].value_counts())

# Keep only barcodes that successfully merged to the long-read library.
# Rows with insert_type == "unknown" are SR barcodes that could not be
# matched to any LR library entry and have no valid insert information.
library_barcodes = merged[merged["insert_type"].isin(["aligned", "empty_insert"])].copy()
print(f"\nLibrary barcodes (aligned + empty_insert): {len(library_barcodes):,}")

## Figure 1C -- Short read frequencies

In [ ]:
sr_freqs = merged.explode(['uncorrected_bc', 'freq', "bl_freq"])
sr_freqs.shape

In [ ]:
sr_freqs.sort_values("freq", ascending=False, inplace=True)
sr_freqs["bc_rank"] = [i for i in range(len(sr_freqs))]
sr_freqs.head()

In [ ]:
fig, ax = plt.subplots(figsize=(3, 3))
expected_median_freq = 1/len(sr_freqs)
n_barcodes = len(sr_freqs)

plt.yscale("log")
plt.axhline(expected_median_freq, color="gray", linestyle="--", linewidth=2, label="Expected med.")
plt.axhline(sr_freqs["freq"].median(), color="black", linestyle="--", linewidth=2, label="Observed med.")
sns.lineplot(data=sr_freqs, x="bc_rank", y="freq")
plt.xlabel("Barcode Rank")
plt.ylabel("Barcode Frequency")
plt.legend(fontsize=10)
ax.set_xticks([1, n_barcodes])
ax.set_xticklabels([1, n_barcodes])
plt.show()
fig.savefig(RESULTS_DIR / "figure1c_library_evenness.pdf", bbox_inches="tight")


print(sr_freqs["freq"].quantile(0.05))
print(sr_freqs["freq"].quantile(0.95))
print(sr_freqs["freq"].quantile(0.95) / sr_freqs["freq"].quantile(0.05))
print(sr_freqs["freq"].median())
print(expected_median_freq)
print(sr_freqs["freq"].describe())



In [ ]:
sr_freqs["freq"].sort_values(ascending=False).head(10)

In [ ]:
# Aligned inserts, one row per unique barcode
aligned = (
    library_barcodes[library_barcodes["insert_type"] == "aligned"]
    .drop_duplicates(subset=["bc_sequence"])
    .copy()
)
aligned["insert_length"] = aligned["insert_sequence"].apply(lambda x: len(x))


# Normalise gene_ids_fully_covered to a plain Python list (handles list, ndarray, None, NaN)
def _to_list(x):
    if isinstance(x, (list, np.ndarray)):
        return [g for g in x if g is not None and g == g]  # drop None / NaN entries
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return []
    return [x]

aligned["gene_ids_fc"] = aligned["gene_ids_fully_covered"].apply(_to_list)
aligned["n_genes_fc"]  = aligned["gene_ids_fc"].apply(len)

print(f"\nAligned inserts (unique barcodes): {len(aligned):,}")
print(f"Median insert length:              {aligned['insert_length'].median():,.0f} bp")
print(f"Median aligned length:             {aligned['aligned_length'].median():,.0f} bp")
print(f"Median genes fully covered:        {aligned['n_genes_fc'].median():.1f}")

In [ ]:
# Load gene information; drop pseudogenes
info_cols = ['pioneer_genome', 'species_name', 'gene_id', 'locus_tag', 'gene_bioname', 'gene_biotype', 'gene_chrom', 'gene_strand', 'gene_start', 'gene_end', 'cds_length', 'protein_id']
gene_info = pd.read_parquet(GENE_INFO_PATH)[info_cols]
gene_info = gene_info[gene_info["gene_biotype"] != "pseudogene"].copy()
print(f"Non-pseudogene genes: {len(gene_info):,}")
print(gene_info.columns.tolist())
gene_info.head(3)

In [ ]:
# Per-gene insert counts derived from gene_ids_fully_covered
inserts_per_gene = (
    aligned[["bc_sequence", "gene_ids_fc"]]
    .explode("gene_ids_fc")
    .dropna(subset=["gene_ids_fc"])
    .query("gene_ids_fc != ''")
    .rename(columns={"gene_ids_fc": "gene_id"})
    .groupby("gene_id")["bc_sequence"]
    .nunique()
    .rename("n_inserts")
    .reset_index()
)

# Merge with full gene list — genes absent from any insert get n_inserts = 0
gene_coverage = gene_info.merge(inserts_per_gene, on="gene_id", how="left")
gene_coverage["n_inserts"] = gene_coverage["n_inserts"].fillna(0).astype(int)
gene_coverage["covered"]   = gene_coverage["n_inserts"] >= 1
gene_coverage["covered_3x"] = gene_coverage["n_inserts"] >= 3

print(f"Genes with ≥1 fully-covering insert: {gene_coverage['covered'].sum():,} / "
      f"{len(gene_coverage):,} ({100 * gene_coverage['covered'].mean():.1f}%)")

print(f"Genes with ≥3 fully-covering inserts: {gene_coverage['covered_3x'].sum():,} / "
      f"{len(gene_coverage):,} ({100 * gene_coverage['covered_3x'].mean():.1f}%)")


### Table 1
Per-genome counts of unique aligned barcodes, empty insert count, median insert length,
median number of fully-covering inserts per gene, and fraction of non-pseudogene genes
covered by ≥1 insert.

In [ ]:
# Identify the genome-level grouping column in both the merged data and gene_info
# The library_dataframe uses 'species_full'; gene_info may use 'pioneer_genome'
SPECIES_COL_INSERTS = "species_full"    # adjust if column name differs in merged parquet
SPECIES_COL_GENES   = "pioneer_genome"  # adjust if column name differs in gene_info

# Per-genome insert stats (already filtered to aligned + deduplicated)
per_genome_inserts = (
    aligned
    .groupby(SPECIES_COL_INSERTS)
    .agg(
        n_inserts=("bc_sequence", "nunique"),
        median_insert_length=("aligned_length", "median"),
    )
    .reset_index()
    .rename(columns={SPECIES_COL_INSERTS: "pioneer_genome"})
)

# Empty inserts — SR barcodes that merged to an empty-insert LR entry
n_empty = int(
    library_barcodes[library_barcodes["insert_type"] == "empty_insert"]["bc_sequence"].nunique()
)

# Per-genome gene coverage stats
per_genome_cov = (
    gene_coverage
    .groupby(SPECIES_COL_GENES)
    .agg(
        n_genes=("gene_id", "count"),
        n_covered=("covered", "sum"),
        median_inserts_per_gene=("n_inserts", "median"),
    )
    .assign(pct_covered=lambda d: d["n_covered"] / d["n_genes"])
    .reset_index()
    .rename(columns={SPECIES_COL_GENES: "pioneer_genome"})
)

# Combine
summary = per_genome_inserts.merge(per_genome_cov, on="pioneer_genome", how="outer")
summary["species_name"] = summary["pioneer_genome"].apply(
    lambda s: " ".join(s.split("_")[:2]) if isinstance(s, str) else s
)
summary = summary.sort_values("species_name").reset_index(drop=True)

# Empty inserts row
empty_row = pd.DataFrame([{
    "pioneer_genome": None, "species_name": "Empty inserts",
    "n_inserts": n_empty, "median_insert_length": float("nan"),
    "median_inserts_per_gene": float("nan"), "pct_covered": float("nan"),
}])

# Total row
total_row = pd.DataFrame([{
    "pioneer_genome": None, "species_name": "Total (library)",
    "n_inserts": int(aligned["bc_sequence"].nunique()),
    "median_insert_length": aligned["aligned_length"].median(),
    "median_inserts_per_gene": gene_coverage["n_inserts"].median(),
    "pct_covered": gene_coverage["covered"].mean(),
}])

summary = pd.concat([summary, empty_row, total_row], ignore_index=True)
print(summary[["species_name", "n_inserts", "median_insert_length", "median_inserts_per_gene", "pct_covered"]])

In [ ]:
tbl = summary[["species_name", "n_inserts", "median_insert_length", "median_inserts_per_gene", "pct_covered"]].copy()

gt_tbl = (
    GT(tbl, rowname_col="species_name")
    .tab_header(title="Library characterization summary")
    .fmt_integer(columns=["n_inserts"])
    .fmt_number(columns=["median_insert_length"], decimals=0)
    .fmt_number(columns=["median_inserts_per_gene"], decimals=1)
    .fmt_percent(columns=["pct_covered"], decimals=1)
    .sub_missing(missing_text="—")
    .cols_label(
        n_inserts="# Inserts",
        median_insert_length="Med. len. (bp)",
        median_inserts_per_gene="Med. inserts/gene",
        pct_covered="Genes ≥1 insert",
    )
    .tab_options(column_labels_font_size=12, table_font_size=12)
)


In [ ]:
import weasyprint

weasyprint.HTML(string=gt_tbl.as_raw_html()).write_pdf(RESULTS_DIR / "Table1_library_summary.pdf")

## Figure 1d — Insert length distribution

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3))
sns.histplot(aligned["insert_length"] / 1000, bins=100, color="#4472C4", edgecolor="none", ax=ax)
ax.set_xlabel("Insert length (kb)", fontsize=10)
ax.set_ylabel("Number of inserts", fontsize=10)
ax.axvline(aligned["aligned_length"].median() / 1000, color="black", linestyle="--", linewidth=2)
ax.tick_params(labelsize=10)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
fig.savefig(RESULTS_DIR / "figure1d_insert_length_distribution.pdf", bbox_inches="tight")
plt.show()

aligned["aligned_length"].describe()

## Figure 1e — Genes per insert (fully covered)

Number of non-pseudogene genes fully spanned by each aligned insert, derived from
`gene_ids_fully_covered`. Inserts with ≥10 genes are grouped into a single bin.

In [ ]:
MAX_BIN = 10
counts = aligned["n_genes_fc"].value_counts().sort_index()
plot_counts = pd.concat([
    counts[counts.index < MAX_BIN],
    pd.Series({f"\u2265{MAX_BIN}": int(counts[counts.index >= MAX_BIN].sum())}),
])

fig, ax = plt.subplots(figsize=(4, 3))
ax.bar(range(len(plot_counts)), plot_counts.values, color="#4472C4", edgecolor="none", alpha=0.7)
ax.set_xticks(range(len(plot_counts)))
ax.set_xticklabels(plot_counts.index.astype(str), fontsize=10)
ax.set_xlabel("Number of fully covered genes per insert", fontsize=10)
ax.set_ylabel("Number of inserts", fontsize=10)
ax.tick_params(axis="y", labelsize=10)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
fig.savefig(RESULTS_DIR / "figure1e_n_genes_per_insert.pdf", bbox_inches="tight")
plt.show()

In [ ]:
plot_counts

## Figure 1f — Inserts per gene distribution

Distribution of the number of unique aligned inserts that fully cover each non-pseudogene
gene. Genes absent from `gene_ids_fully_covered` in all inserts are included with n = 0.

In [ ]:
bins = [-1, 0.5, 10, 20, 30, 40, 50, np.inf]
labels = ['0', '1-10', '11-20', '21-30', '31-40', '41-50', '>50']

# 3. Create a new column with the assigned bins
gene_coverage['ranges'] = pd.cut(gene_coverage['n_inserts'], bins=bins, labels=labels, right=True)
gene_coverage.ranges.value_counts()

In [ ]:
n_inserts = gene_coverage["n_inserts"]
n_inserts_median = n_inserts.median()
n_inserts[n_inserts > 80] = 81
gene_covered = pd.cut(n_inserts, bins=[-1,0,2,n_inserts.max()+1], labels=["0", "1-2", ">=3"], right=True)

fig, ax = plt.subplots(figsize=(4, 3))
sns.histplot(x = n_inserts, color="#4472C4", edgecolor="none", ax=ax, stat="count", binwidth=1, hue=gene_covered, palette=["red", "orange", "#4472C4"], alpha=0.7)
ax.set_xlabel("Number of inserts per gene", fontsize=10)
### Rotate x axis labels
ax.set_ylabel("Number of genes", fontsize=10)
ax.axvline(n_inserts_median, color="black", linestyle="--", linewidth=2)
ax.tick_params(labelsize=10)
ax.tick_params(axis='x', labelrotation=45)
ax.set_xticks([0, 10, 20, 30, 40, 50, 60, 70, 80])
ax.set_xticklabels([0, 10, 20, 30, 40, 50, 60, 70, ">80"])
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
fig.savefig(RESULTS_DIR / "figure1f_inserts_per_gene_distribution.pdf", bbox_inches="tight")
plt.show()

print(gene_coverage["n_inserts"].describe())

In [ ]:
gene_coverage["ranges"].value_counts()

In [ ]:
# Per-genome median inserts per gene (non-pseudogene genes only)
if SPECIES_COL_GENES in gene_coverage.columns:
    display(
        gene_coverage
        .groupby(SPECIES_COL_GENES)["n_inserts"]
        .agg(["median", "mean", lambda x: (x >= 1).mean()])
        .rename(columns={"median": "median_inserts", "mean": "mean_inserts", "<lambda_0>": "pct_covered"})
        .sort_values("median_inserts")
    )